# Deep Residual Probabilistic PLS — Interactive Notebook

**Paper**: Ge, Chen & Yu (2025) — *Deep Residual Learning of a Probabilistic Partial Least Squares Model*  
**Journal**: IEEE Transactions on Artificial Intelligence, Vol. 6, No. 11

---

## Input Files Required
Place these in the `data/` folder before running:
- `data/TEP_FaultFree_Testing.csv` — TEP process dataset (56 columns)
- `data/Concrete_Data.csv` — Concrete strength dataset (9 columns)

---
**To run everything at once**, use the terminal instead:
```bash
python run_all.py
```


In [ ]:
import os, sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 110

from data_loader    import load_dataset, TEP_INPUT_COLS, TEP_OUTPUT_COL
from data_loader    import CONCRETE_INPUT_COLS, CONCRETE_OUTPUT_COL
from residual_model import DeepResidualPLS, BaselinePLS
from evaluate       import evaluate_model, compare_models, rmse, r2, mae
from visualize      import (plot_actual_vs_predicted, plot_parity,
                            plot_error_distribution, plot_layerwise_rmse,
                            plot_with_uncertainty, plot_dashboard)

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../results',  exist_ok=True)
print("Imports OK")


## 1. Choose Dataset

Set `DATASET` to either `'tep'` or `'concrete'`.

| Value | CSV file | Inputs | Output |
|---|---|---|---|
| `'tep'` | `TEP_FaultFree_Testing.csv` | `xmeas_1` .. `xmeas_40` (40 cols) | `xmeas_41` |
| `'concrete'` | `Concrete_Data.csv` | Cement, Water, Slag, … (8 cols) | Compressive strength (MPa) |


In [ ]:
# ── SELECT DATASET HERE ─────────────────────────────────────────────────────
DATASET = 'tep'        # change to 'concrete' for the cement dataset

# Model hyperparameters (matching paper settings)
PARAMS = {
    'tep':      dict(layers=3, components=3, final_comp=2),
    'concrete': dict(layers=3, components=4, final_comp=3),
}
cfg = PARAMS[DATASET]
print(f"Dataset  : {DATASET}")
print(f"Layers   : {cfg['layers']}")
print(f"qs/layer : {cfg['components']}")
print(f"qs fusion: {cfg['final_comp']}")


## 2. Load Data

In [ ]:
X_train, X_test, Y_train, Y_test, scaler_X, scaler_Y = load_dataset(
    DATASET, data_dir='../data', test_size=0.5)

print(f"\nX_train : {X_train.shape}  X_test : {X_test.shape}")
print(f"Y_train : {Y_train.shape}  Y_test : {Y_test.shape}")


In [ ]:
# Column names for reference
if DATASET == 'tep':
    print("INPUTS  (40 cols):", TEP_INPUT_COLS[:5], "...", TEP_INPUT_COLS[-2:])
    print("OUTPUT  (1  col) :", TEP_OUTPUT_COL)
else:
    print("INPUTS  (8  cols):", CONCRETE_INPUT_COLS)
    print("OUTPUT  (1  col) :", CONCRETE_OUTPUT_COL)


In [ ]:
# Output distribution
fig, axes = plt.subplots(1,2, figsize=(12,3))
axes[0].plot(Y_train.ravel(), lw=0.8, color='#2c3e50')
axes[0].set_title('Training output (Y_train)'); axes[0].set_xlabel('Sample')
axes[1].hist(Y_train.ravel(), bins=40, color='#2980b9', edgecolor='white', lw=0.3)
axes[1].set_title('Output distribution'); axes[1].set_xlabel('Y value')
plt.tight_layout(); plt.show()


## 3. Train Baseline (Single-Layer) PLS

In [ ]:
baseline = BaselinePLS(n_components=cfg['components'])
baseline.fit(X_train, Y_train)
Y_pred_base = baseline.predict(X_test)
res_base = evaluate_model('Baseline PLS', Y_test, Y_pred_base)


## 4. Train Deep Residual PLS

In [ ]:
deep_model = DeepResidualPLS(
    n_layers           = cfg['layers'],
    n_components_layer = cfg['components'],
    n_components_final = cfg['final_comp'],
)
deep_model.fit(X_train, Y_train)
Y_pred_deep = deep_model.predict(X_test)
res_deep = evaluate_model('Deep Residual PLS', Y_test, Y_pred_deep)


## 5. Layer-wise Comparison

In [ ]:
actual_layers = len(deep_model._layers)
layer_preds = {l: deep_model.predict_layer(X_test, l) for l in range(1, actual_layers+1)}

layer_results = [evaluate_model(f'Layer {l}', Y_test, layer_preds[l])
                 for l in layer_preds]
compare_models([res_base] + layer_results + [res_deep])

pct_rmse = (res_base['rmse'] - res_deep['rmse']) / res_base['rmse'] * 100
print(f"\n  RMSE improvement: {pct_rmse:+.1f}%  (paper reports 10-20%)")


## 6. Visualisations

In [ ]:
# Actual vs Predicted — first 200 samples
n = min(200, len(Y_test))
fig, axes = plt.subplots(2,1, figsize=(14,7), sharex=True)
fig.suptitle(f'Actual vs Predicted — {DATASET.upper()}', fontsize=13, fontweight='bold')
for ax, pred, lbl, clr in [
    (axes[0], Y_pred_base, 'Baseline PLS',      '#e74c3c'),
    (axes[1], Y_pred_deep, 'Deep Residual PLS', '#2980b9'),
]:
    ax.plot(Y_test.ravel()[:n], lw=1.5, color='#2c3e50', label='Actual')
    ax.plot(pred.ravel()[:n], lw=1.0, ls='--', color=clr, alpha=0.9, label=lbl)
    ax.legend(); ax.set_ylabel('Output'); ax.set_title(f'({lbl})')
axes[1].set_xlabel('Sample index')
plt.tight_layout(); plt.show()


In [ ]:
# Parity plot
fig, axes = plt.subplots(1,2, figsize=(12,5))
fig.suptitle('Parity Plot', fontsize=13, fontweight='bold')
yt = Y_test.ravel()
vmin, vmax = yt.min(), yt.max(); mg = (vmax-vmin)*0.05
for ax, pred, title, clr in [
    (axes[0], Y_pred_base, f'Baseline PLS   R²={r2(Y_test,Y_pred_base):.4f}', '#e74c3c'),
    (axes[1], Y_pred_deep, f'Deep PLS       R²={r2(Y_test,Y_pred_deep):.4f}', '#2980b9'),
]:
    ax.scatter(pred.ravel(), yt, alpha=0.35, s=15, color=clr)
    diag = np.linspace(vmin-mg, vmax+mg, 100)
    ax.plot(diag, diag, 'k--', lw=1.2, label='Perfect')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(title); ax.set_aspect('equal','box'); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# Layer-wise RMSE bar chart
layer_rmses = [rmse(Y_test, layer_preds[l]) for l in layer_preds]
lbls = [f'Layer {l}' for l in layer_preds] + ['Baseline', 'Deep PLS']
vals = layer_rmses + [res_base['rmse'], res_deep['rmse']]
clrs = ['#8e44ad','#27ae60','#f39c12'][:actual_layers] + ['#e74c3c','#2980b9']

fig, ax = plt.subplots(figsize=(10,5))
bars = ax.bar(lbls, vals, color=clrs, edgecolor='white', width=0.6)
bars[-1].set_edgecolor('black'); bars[-1].set_linewidth(2)
for bar,v in zip(bars,vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+max(vals)*0.01,
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('RMSE'); ax.set_title('Layer-wise RMSE (lower is better)')
plt.tight_layout(); plt.show()


In [ ]:
# Save full dashboard
plot_dashboard(Y_test, Y_pred_base, Y_pred_deep, layer_preds,
               res_base, res_deep,
               dataset_name=DATASET.upper(), save_dir='../outputs')
print("Dashboard saved to outputs/")


## 7. Save Results

In [ ]:
import pickle, numpy as np

prefix = f'../results/{DATASET}'
np.save(f'{prefix}_Y_test.npy',      Y_test)
np.save(f'{prefix}_Y_pred_base.npy', Y_pred_base)
np.save(f'{prefix}_Y_pred_deep.npy', Y_pred_deep)
for l in layer_preds:
    np.save(f'{prefix}_Y_pred_layer{l}.npy', layer_preds[l])
with open(f'{prefix}_deep_model.pkl',    'wb') as f: pickle.dump(deep_model, f)
with open(f'{prefix}_baseline_model.pkl','wb') as f: pickle.dump(baseline,   f)

with open(f'{prefix}_summary.txt','w') as f:
    f.write(f"Dataset: {DATASET}\n"
            f"Layers: {actual_layers}\n"
            f"Baseline RMSE={res_base['rmse']:.5f}  R2={res_base['r2']:.5f}\n"
            f"Deep PLS RMSE={res_deep['rmse']:.5f}  R2={res_deep['r2']:.5f}\n"
            f"RMSE improvement: {pct_rmse:+.1f}%\n")
print("All results saved to results/")
